# A3 Window C — seat placement prototype (re-run notebook)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Starive/A3-for-ARCH7476/blob/main/a3_seat_C.ipynb)

**How to run:** Runtime → Run all. Nothing needs to be uploaded or downloaded.

**Decision supported:** which of six 2.0 × 1.0 m seat placements in Window C best supports short stays while keeping pedestrian clearance?

**What it does:** runs 36 scenarios (6 placements × 3 corridor half-widths × 2 edge proxies), applies a rule stated in advance, runs 25 checks and an independent GeoPandas cross-check, and draws the figures.

**Limits:** local-metre schematic of an assumed 6 × 6 m window. No map coordinates are loaded. Window C, the edge proxies, w, c₁, c₂ and s are researcher assumptions; the results are not evidence of pedestrian flow or comfort.

## 1. Environment

In [ ]:
import importlib.util, subprocess, sys, platform
missing = [m for m in ['matplotlib', 'geopandas', 'shapely', 'pandas'] if importlib.util.find_spec(m) is None]
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + missing, check=True)
print('python', platform.python_version())
for m in ['matplotlib', 'geopandas', 'shapely', 'pandas']:
    print(m, importlib.import_module(m).__version__)

## 2. Code and inputs
If the notebook sits inside the repository folder, those files are used. Otherwise the embedded copies are written to `A3_rerun/`.

In [ ]:
from pathlib import Path
FILES = {"a3src/a3_workflow.py": "\"\"\"A3 Window C workflow core.\n\nLocal-metre schematic frame: x = distance inward from the east edge proxy (0..window),\ny = distance south from the north edge proxy (0..window). No map coordinates are loaded.\nOne 2 x 1 m seat is tested at six alternative placements. Each seat keeps the same position\nin space under both edge proxies; only the edge used as the movement proxy changes.\n\nThe core uses exact interval arithmetic and needs only the Python standard library.\n`geopandas_cross_check` repeats the overlap test with geopandas.overlay as an independent check.\n\"\"\"\nimport csv\nimport json\nfrom pathlib import Path\n\nEPS = 1e-9\n\n\ndef load_inputs(root):\n    root = Path(root)\n    cfg = json.loads((root / \"inputs\" / \"config.json\").read_text(encoding=\"utf8\"))\n    with open(root / \"inputs\" / \"variants.csv\", encoding=\"utf8\", newline=\"\") as f:\n        variants = [dict(r, centre_from_east_m=float(r[\"centre_from_east_m\"])) for r in csv.DictReader(f)]\n    return cfg, variants\n\n\ndef seat_rect(variant, cfg, centre_from_east=None):\n    \"\"\"Seat rectangle (x0, x1, y0, y1). 'parallel' = long side N-S, along the east edge.\"\"\"\n    xc = variant[\"centre_from_east_m\"] if centre_from_east is None else centre_from_east\n    yc = cfg[\"seat_centre_from_north_m\"]\n    if variant[\"orientation\"] == \"parallel\":\n        hx, hy = cfg[\"seat_short_m\"] / 2, cfg[\"seat_long_m\"] / 2\n    else:\n        hx, hy = cfg[\"seat_long_m\"] / 2, cfg[\"seat_short_m\"] / 2\n    return (xc - hx, xc + hx, yc - hy, yc + hy)\n\n\ndef _overlap(a0, a1, b0, b1):\n    return max(0.0, min(a1, b1) - max(a0, b0))\n\n\ndef evaluate(rect, edge, w, cfg, c1=None, c2=None):\n    \"\"\"Corridor overlap (m2) and proposed band share (%) of one seat for one edge proxy and width.\"\"\"\n    c1 = cfg[\"c1_m\"] if c1 is None else c1\n    c2 = cfg[\"c2_m\"] if c2 is None else c2\n    x0, x1, y0, y1 = rect\n    d0, d1, along = (x0, x1, y1 - y0) if edge == \"east\" else (y0, y1, x1 - x0)\n    W = cfg[\"window_m\"]\n    area = (x1 - x0) * (y1 - y0)\n    overlap = _overlap(d0, d1, 0.0, w) * along\n    band = _overlap(d0, d1, min(w + c1, W), min(w + c2, W)) * along\n    return overlap, 100.0 * band / area\n\n\ndef passes(overlap, share, cfg, s=None):\n    s = cfg[\"s_pct\"] if s is None else s\n    return overlap <= EPS and share >= s - EPS\n\n\ndef run_scenarios(cfg, variants):\n    rows = []\n    for edge in cfg[\"edge_proxies\"]:\n        for w in cfg[\"widths_m\"]:\n            for v in variants:\n                rect = seat_rect(v, cfg)\n                overlap, share = evaluate(rect, edge, w, cfg)\n                x0, x1, y0, y1 = rect\n                rows.append({\n                    \"variant\": v[\"variant\"], \"position\": v[\"position\"], \"orientation\": v[\"orientation\"],\n                    \"edge_proxy\": edge, \"w_m\": w,\n                    \"corridor_overlap_m2\": round(overlap, 4), \"band_share_pct\": round(share, 2),\n                    \"seat_area_m2\": round((x1 - x0) * (y1 - y0), 4),\n                    \"inside_window\": x0 >= -EPS and y0 >= -EPS and x1 <= cfg[\"window_m\"] + EPS and y1 <= cfg[\"window_m\"] + EPS,\n                    \"geometry\": \"conflict\" if overlap > EPS else \"clear\",\n                    \"passes_rule\": passes(overlap, share, cfg),\n                })\n    return rows\n\n\ndef variant_summary(rows, cfg):\n    out = []\n    for vid in sorted({r[\"variant\"] for r in rows}):\n        mine = [r for r in rows if r[\"variant\"] == vid]\n        rec = {\"variant\": vid}\n        for edge in cfg[\"edge_proxies\"]:\n            e = [r for r in mine if r[\"edge_proxy\"] == edge]\n            rec[f\"{edge}_share_min_pct\"] = min(r[\"band_share_pct\"] for r in e)\n            rec[f\"{edge}_share_max_pct\"] = max(r[\"band_share_pct\"] for r in e)\n            rec[f\"{edge}_max_overlap_m2\"] = max(r[\"corridor_overlap_m2\"] for r in e)\n            rec[f\"{edge}_passes_all_widths\"] = all(r[\"passes_rule\"] for r in e)\n        rec[\"passes_all_36_conditions\"] = all(r[\"passes_rule\"] for r in mine)\n        out.append(rec)\n    return out\n\n\ndef position_sweep(cfg, orientation=\"parallel\", edge=\"east\", start=1.5, stop=4.5):\n    \"\"\"Move one seat in steps along x and test every tested width (exploratory).\"\"\"\n    step = cfg[\"sweep_step_m\"]\n    out, n = [], int(round((stop - start) / step))\n    for k in range(n + 1):\n        xc = round(start + k * step, 4)\n        rect = seat_rect({\"orientation\": orientation, \"centre_from_east_m\": xc}, cfg)\n        res = [evaluate(rect, edge, w, cfg) for w in cfg[\"widths_m\"]]\n        out.append({\"orientation\": orientation, \"edge_proxy\": edge, \"centre_from_east_m\": xc,\n                    \"share_min_pct\": round(min(s for _, s in res), 2),\n                    \"max_overlap_m2\": round(max(o for o, _ in res), 4),\n                    \"passes_all_widths\": all(passes(o, s, cfg) for o, s in res)})\n    return out\n\n\ndef width_curve(variant, cfg, edge=\"east\", lo=0.5, hi=2.1, step=0.01):\n    out, n = [], int(round((hi - lo) / step))\n    rect = seat_rect(variant, cfg)\n    for k in range(n + 1):\n        w = round(lo + k * step, 4)\n        overlap, share = evaluate(rect, edge, w, cfg)\n        out.append({\"variant\": variant[\"variant\"], \"edge_proxy\": edge, \"w_m\": w,\n                    \"band_share_pct\": round(share, 3), \"corridor_overlap_m2\": round(overlap, 4),\n                    \"passes_rule\": passes(overlap, share, cfg)})\n    return out\n\n\ndef pass_range(curve):\n    ws = [r[\"w_m\"] for r in curve if r[\"passes_rule\"]]\n    return (min(ws), max(ws)) if ws else (None, None)\n\n\ndef internal_checks(cfg, variants, rows):\n    \"\"\"25 checks. Expected values were written by hand before the code was run.\"\"\"\n    V = {v[\"variant\"]: v for v in variants}\n    get = lambda vid, edge, w: next(r for r in rows if r[\"variant\"] == vid and r[\"edge_proxy\"] == edge and abs(r[\"w_m\"] - w) < EPS)\n    checks = []\n\n    def add(name, expected, observed, ok):\n        checks.append({\"n\": len(checks) + 1, \"check\": name, \"expected\": expected, \"observed\": observed,\n                       \"status\": \"PASS\" if ok else \"FAIL\"})\n\n    add(\"Scenario count\", \"6 x 3 x 2 = 36\", len(rows), len(rows) == 36)\n    for vid in sorted(V):\n        a = get(vid, \"east\", 1.0)[\"seat_area_m2\"]\n        add(f\"{vid} seat area\", \"2.00 m2\", f\"{a:.2f} m2\", abs(a - 2.0) < 1e-6)\n    for vid in sorted(V):\n        ins = get(vid, \"east\", 1.0)[\"inside_window\"]\n        add(f\"{vid} inside Window C\", \"True\", str(ins), ins)\n    known = [(\"S1\", \"east\", 1.0, \"band_share_pct\", 80.0, \"seat x 1.7-2.7 in band 1.3-2.5: 0.8 of 1.0\"),\n             (\"S2\", \"east\", 1.0, \"band_share_pct\", 60.0, \"seat x 1.2-3.2 in band 1.3-2.5: 1.2 of 2.0\"),\n             (\"S1\", \"east\", 0.75, \"band_share_pct\", 55.0, \"band 1.05-2.25: 0.55 of 1.0\"),\n             (\"S1\", \"east\", 1.5, \"band_share_pct\", 90.0, \"band 1.8-3.0: 0.9 of 1.0\"),\n             (\"S2\", \"east\", 1.5, \"corridor_overlap_m2\", 0.30, \"seat x 1.2-1.5 inside corridor: 0.3 m x 1.0 m\"),\n             (\"S1\", \"north\", 1.0, \"band_share_pct\", 25.0, \"seat y 2.0-4.0 in band 1.3-2.5: 0.5 of 2.0\"),\n             (\"S4\", \"east\", 1.0, \"band_share_pct\", 15.0, \"seat x 2.2-4.2 in band 1.3-2.5: 0.3 of 2.0\")]\n    for vid, edge, w, col, exp, why in known:\n        obs = get(vid, edge, w)[col]\n        add(f\"Known case {vid} / {edge} / w={w:.2f}: {col}\", f\"{exp} ({why})\", obs, abs(obs - exp) < 1e-6)\n    shares = [r[\"band_share_pct\"] for r in rows]\n    add(\"All band shares within 0-100 %\", \"True\", f\"{min(shares)} to {max(shares)}\", min(shares) >= 0 and max(shares) <= 100)\n    ovs = [r[\"corridor_overlap_m2\"] for r in rows]\n    add(\"All overlaps within 0-2.00 m2\", \"True\", f\"{min(ovs)} to {max(ovs)}\", min(ovs) >= 0 and max(ovs) <= 2.0)\n    none075 = all(r[\"corridor_overlap_m2\"] == 0 for r in rows if r[\"edge_proxy\"] == \"east\" and abs(r[\"w_m\"] - 0.75) < EPS)\n    add(\"No east-proxy overlap at w = 0.75 m\", \"True\", str(none075), none075)\n    sweep_pass = [r[\"centre_from_east_m\"] for r in position_sweep(cfg) if r[\"passes_all_widths\"]]\n    add(\"Position sweep (parallel, east): passing centres\", \"2.00 to 2.15 m\",\n        f\"{min(sweep_pass):.2f} to {max(sweep_pass):.2f} m\" if sweep_pass else \"none\",\n        bool(sweep_pass) and abs(min(sweep_pass) - 2.0) < 1e-6 and abs(max(sweep_pass) - 2.15) < 1e-6)\n    lo, hi = pass_range(width_curve(V[\"S1\"], cfg))\n    add(\"S1 east-proxy pass range of w\", \"0.80 to 1.70 m\", f\"{lo:.2f} to {hi:.2f} m\",\n        lo is not None and abs(lo - 0.80) < 1e-6 and abs(hi - 1.70) < 1e-6)\n    return checks\n\n\ndef geopandas_cross_check(cfg, variants, rows):\n    \"\"\"Independent overlap test with geopandas.overlay for every east-proxy scenario.\"\"\"\n    try:\n        import geopandas as gpd\n        from shapely.geometry import box\n    except ImportError:\n        return {\"status\": \"SKIPPED\", \"detail\": \"geopandas not installed in this environment\"}\n    try:\n        W = cfg[\"window_m\"]\n        seats = gpd.GeoDataFrame({\"variant\": [v[\"variant\"] for v in variants]},\n                                 geometry=[box(r[0], r[2], r[1], r[3]) for r in (seat_rect(v, cfg) for v in variants)])\n        worst, s2_case = 0.0, None\n        for w in cfg[\"widths_m\"]:\n            corridor = gpd.GeoDataFrame({\"corridor\": [1]}, geometry=[box(0.0, 0.0, w, W)])\n            inter = gpd.overlay(seats, corridor, how=\"intersection\")\n            areas = dict(zip(inter[\"variant\"], inter.geometry.area)) if len(inter) else {}\n            for v in variants:\n                gp = float(areas.get(v[\"variant\"], 0.0))\n                core = next(r[\"corridor_overlap_m2\"] for r in rows\n                            if r[\"variant\"] == v[\"variant\"] and r[\"edge_proxy\"] == \"east\" and abs(r[\"w_m\"] - w) < EPS)\n                worst = max(worst, abs(gp - core))\n                if v[\"variant\"] == \"S2\" and abs(w - 1.5) < EPS:\n                    s2_case = gp\n        ok = worst < 1e-6 and s2_case is not None and abs(s2_case - 0.30) < 1e-6\n        return {\"status\": \"PASS\" if ok else \"FAIL\",\n                \"detail\": f\"S2 / east / w=1.50 m overlap = {s2_case:.2f} m2; max difference from core over 18 east scenarios = {worst:.2e}\"}\n    except Exception as exc:  # report, do not hide\n        return {\"status\": \"ERROR\", \"detail\": f\"{type(exc).__name__}: {exc}\"}\n\n\ndef write_csv(path, rows):\n    with open(path, \"w\", encoding=\"utf8\", newline=\"\") as f:\n        wr = csv.DictWriter(f, fieldnames=list(rows[0].keys()))\n        wr.writeheader()\n        wr.writerows(rows)\n","a3_osm_anchor.py": "\"\"\"A3 Window C: run the whole workflow.\n\nUsage:  python a3_osm_anchor.py\nWrites outputs/ (CSV tables, figures, check log, versions). Edit inputs/config.json to test other assumptions.\nThe file name is kept from the first draft; this version loads no OpenStreetMap coordinates\n(the OSM way ID in config.json is provenance only).\n\"\"\"\nimport platform\nimport sys\nfrom pathlib import Path\n\nROOT = Path(__file__).resolve().parent\nsys.path.insert(0, str(ROOT))\nfrom a3src import a3_workflow as wf  # noqa: E402\n\nimport matplotlib  # noqa: E402\nmatplotlib.use(\"Agg\")\nimport matplotlib.pyplot as plt  # noqa: E402\nfrom matplotlib.patches import Rectangle  # noqa: E402\n\nOUT = ROOT / \"outputs\"\nOUT.mkdir(exist_ok=True)\ncfg, variants = wf.load_inputs(ROOT)\nV = {v[\"variant\"]: v for v in variants}\n\n# 1. scenarios, summaries, sweep\nrows = wf.run_scenarios(cfg, variants)\nwf.write_csv(OUT / \"a3_scenarios.csv\", rows)\nsummary = wf.variant_summary(rows, cfg)\nwf.write_csv(OUT / \"a3_variant_summary.csv\", summary)\nsweep = wf.position_sweep(cfg)\nwf.write_csv(OUT / \"a3_position_sweep.csv\", sweep)\ncurves = {vid: wf.width_curve(V[vid], cfg) for vid in (\"S1\", \"S2\")}\nwf.write_csv(OUT / \"a3_width_sensitivity.csv\", curves[\"S1\"] + curves[\"S2\"])\n\n# 2. checks\nchecks = wf.internal_checks(cfg, variants, rows)\ncross = wf.geopandas_cross_check(cfg, variants, rows)\nlog = checks + [{\"n\": \"GP\", \"check\": \"GeoPandas overlay cross-check (east proxy)\", \"expected\": \"S2 / w=1.50 m = 0.30 m2; no difference from core\",\n                 \"observed\": cross[\"detail\"], \"status\": cross[\"status\"]}]\nwf.write_csv(OUT / \"a3_checks.csv\", log)\nn_pass = sum(c[\"status\"] == \"PASS\" for c in checks)\n\n# 3. versions\nvers = [\"python \" + platform.python_version(), \"matplotlib \" + matplotlib.__version__]\nfor mod in (\"geopandas\", \"shapely\", \"pandas\", \"pyproj\"):\n    try:\n        vers.append(f\"{mod} {__import__(mod).__version__}\")\n    except ImportError:\n        vers.append(f\"{mod} not installed\")\n(OUT / \"versions.txt\").write_text(\"\\n\".join(vers) + \"\\n\", encoding=\"utf8\")\n\n# 4. figures\nGREEN, TEAL, TAN, ORANGE, RULE = \"#1f5a4e\", \"#9fbfb0\", \"#dcc3a5\", \"#c8623f\", \"#d9532b\"\nW, BW, C1, C2, S = cfg[\"window_m\"], cfg[\"base_width_m\"], cfg[\"c1_m\"], cfg[\"c2_m\"], cfg[\"s_pct\"]\nget = lambda vid, edge, w: next(r for r in rows if r[\"variant\"] == vid and r[\"edge_proxy\"] == edge and abs(r[\"w_m\"] - w) < 1e-9)\n\nfig, axes = plt.subplots(2, 3, figsize=(13.6, 9.4))\nfor ax, v in zip(axes.flat, variants):\n    r = get(v[\"variant\"], \"east\", BW)\n    x0, x1, y0, y1 = wf.seat_rect(v, cfg)\n    ax.add_patch(Rectangle((-W, -W), W, W, fill=False, ec=\"#4b6b62\", ls=\"--\", lw=1))\n    ax.add_patch(Rectangle((-BW, -W), BW, W, fc=TAN, ec=\"none\", alpha=0.65))\n    ax.add_patch(Rectangle((-(BW + C2), -W), C2 - C1, W, fc=TEAL, ec=\"none\", alpha=0.5))\n    ax.plot([0, 0], [0, -W], color=GREEN, lw=3)\n    conflict = r[\"corridor_overlap_m2\"] > 0\n    ax.add_patch(Rectangle((-x1, -y1), x1 - x0, y1 - y0, fc=ORANGE, ec=\"#8f1d1d\" if conflict else ORANGE, lw=2.5 if conflict else 1.5, alpha=0.7))\n    ax.set_title(f'{v[\"variant\"]} · {v[\"position\"].lower()} / {v[\"orientation\"]}\\n'\n                 f'Band share {r[\"band_share_pct\"]:.0f}% · overlap {r[\"corridor_overlap_m2\"]:.2f} m²', fontsize=11, color=\"#1f3b34\")\n    ax.set_xlim(-W - 0.5, 0.45); ax.set_ylim(-W - 0.5, 0.4); ax.set_aspect(\"equal\")\n    ax.set_xlabel(\"Local x (m) · east edge at x=0\", fontsize=8.5); ax.set_ylabel(\"Local y (m) · north edge at y=0\", fontsize=8.5)\n    ax.grid(alpha=0.12)\nfig.suptitle(f\"Window C / six ALTERNATIVE placements of ONE 2 × 1 m seat\\nAssumed 6 × 6 m geometry; east edge proxy, w = {BW:.1f} m\",\n             fontsize=15, fontweight=\"bold\", color=GREEN)\nfig.text(0.5, 0.015, \"Schematic geometry only • green = proposed flow-adjacent band • tan = assumed corridor • orange = seat\",\n         ha=\"center\", fontsize=9.5, color=\"#5f786f\")\nfig.tight_layout(rect=(0, 0.03, 1, 0.93), h_pad=2.2)\nfig.savefig(OUT / \"fig_six_placements.png\", dpi=160); plt.close(fig)\n\nids = [v[\"variant\"] for v in variants]\nfor edge in cfg[\"edge_proxies\"]:\n    base = [get(v, edge, BW) for v in ids]\n    shares = [r[\"band_share_pct\"] for r in base]\n    lead = ids[max(range(len(ids)), key=lambda i: shares[i])]\n    fig, ax = plt.subplots(figsize=(10, 4.9))\n    ax.bar(ids, shares, color=[GREEN if r[\"passes_rule\"] else \"#5f7d73\" for r in base], width=0.66)\n    for i, s in enumerate(shares):\n        ax.text(i, s + 1.5, f\"{s:.0f}%\", ha=\"center\", fontsize=10, color=\"#2a3d38\")\n    ax.axhline(S, color=RULE, lw=2, ls=\"--\", label=f\"Assumed research threshold s = {S:.0f}%\")\n    ax.set_ylim(0, 105); ax.set_ylabel(\"Proposed flow-adjacent band share (%)\")\n    title = (f\"Base-case {lead} leads; proxy threshold is NOT a comfort standard\" if max(shares) >= S\n             else f\"No variant reaches s = {S:.0f}% under the {edge} proxy\")\n    ax.set_title(title, fontweight=\"bold\", color=GREEN)\n    ax.legend(frameon=False, loc=\"upper right\", fontsize=9); ax.spines[[\"top\", \"right\"]].set_visible(False)\n    fig.text(0.5, 0.02, f\"{edge.capitalize()} edge proxy · corridor half-width w = {BW:.2f} m · c₁={C1:.2f}m, c₂={C2:.2f}m\", ha=\"center\", fontsize=9, color=\"#5f786f\")\n    fig.tight_layout(rect=(0, 0.05, 1, 1))\n    fig.savefig(OUT / f\"fig_base_comparison_{edge}.png\", dpi=160); plt.close(fig)\n\nlo, hi = wf.pass_range(curves[\"S1\"])\nfig, ax = plt.subplots(figsize=(10, 5.2))\nif lo is not None:\n    ax.axvspan(lo, hi, color=\"#e7efea\", zorder=0)\n    ax.axvline(lo, color=\"#b08a3c\", lw=1, ls=\":\"); ax.axvline(hi, color=\"#b08a3c\", lw=1, ls=\":\")\n    ax.text(lo + 0.02, 6, \"S1 conditional pass range\\n(excluding geometric overlaps)\", fontsize=8.5, color=GREEN)\nfor vid, col in ((\"S1\", GREEN), (\"S2\", ORANGE)):\n    ax.plot([r[\"w_m\"] for r in curves[vid]], [r[\"band_share_pct\"] for r in curves[vid]], color=col, lw=2.2, label=f\"{vid} band share\")\n    for w in cfg[\"widths_m\"]:\n        ax.plot(w, get(vid, \"east\", w)[\"band_share_pct\"], \"o\", color=col, ms=6)\nax.axhline(S, color=\"#a8522f\", lw=1.6, ls=\"--\", label=f\"Assumed s = {S:.0f}%\")\nax.set_xlim(0.5, 2.1); ax.set_ylim(0, 108)\nax.set_xlabel(\"Assumed corridor half-width w (m)\"); ax.set_ylabel(\"Proposed band share (%)\")\nax.set_title(f\"Sensitivity: S1 clears s only for {lo:.2f} ≤ w ≤ {hi:.2f} m (east proxy)\" if lo is not None else \"Sensitivity: S1 never clears s (east proxy)\",\n             fontweight=\"bold\", color=GREEN)\nax.legend(frameon=False, loc=\"upper left\", fontsize=9); ax.spines[[\"top\", \"right\"]].set_visible(False)\nfig.tight_layout(); fig.savefig(OUT / \"fig_width_sensitivity.png\", dpi=160); plt.close(fig)\n\n# 5. report\nprint(f\"Scenarios: {len(rows)} rows -> outputs/a3_scenarios.csv\")\nprint(f\"Internal checks: {n_pass} of {len(checks)} PASS\")\nfor c in checks:\n    if c[\"status\"] != \"PASS\":\n        print(\"  FAIL:\", c[\"check\"], \"| expected\", c[\"expected\"], \"| observed\", c[\"observed\"])\nprint(f\"GeoPandas cross-check: {cross['status']} - {cross['detail']}\")\npassing = [r[\"centre_from_east_m\"] for r in sweep if r[\"passes_all_widths\"]]\nprint(\"Variants passing all 36 conditions:\", [s[\"variant\"] for s in summary if s[\"passes_all_36_conditions\"]] or \"none\")\nprint(\"Position sweep (parallel seat, east proxy) passes all three widths at centres:\",\n      f\"{min(passing):.2f}-{max(passing):.2f} m\" if passing else \"none\")\nprint(\"These are geometric screening results under assumed inputs, not evidence of pedestrian flow or comfort.\")\nif n_pass != len(checks) or cross[\"status\"] in (\"FAIL\", \"ERROR\"):\n    sys.exit(1)\n","check_reproduction.py": "\"\"\"Compare a fresh run with the reference results. Usage: python check_reproduction.py (after a3_osm_anchor.py).\"\"\"\nimport csv\nimport sys\nfrom pathlib import Path\n\nROOT = Path(__file__).resolve().parent\nTOL = 0.01\n\n\ndef load(path):\n    with open(path, encoding=\"utf8\", newline=\"\") as f:\n        return {(r[\"variant\"], r[\"edge_proxy\"], f'{float(r[\"w_m\"]):.2f}'): r for r in csv.DictReader(f)}\n\n\nnew = load(ROOT / \"outputs\" / \"a3_scenarios.csv\")\nref = load(ROOT / \"reference\" / \"a3_scenarios_reference.csv\")\nproblems = [] if set(new) == set(ref) else [f\"scenario keys differ: {len(new)} new vs {len(ref)} reference\"]\nfor key in sorted(set(new) & set(ref)):\n    for col in (\"corridor_overlap_m2\", \"band_share_pct\"):\n        if abs(float(new[key][col]) - float(ref[key][col])) > TOL:\n            problems.append(f\"{key} {col}: got {new[key][col]}, reference {ref[key][col]}\")\n    if new[key][\"passes_rule\"].strip().lower() != ref[key][\"passes_rule\"].strip().lower():\n        problems.append(f\"{key} passes_rule differs\")\nif problems:\n    print(\"NOT REPRODUCED (expected if inputs/config.json or variants.csv was edited):\")\n    for p in problems:\n        print(\"  -\", p)\n    sys.exit(1)\nprint(f\"REPRODUCED: all {len(ref)} scenarios match the reference within {TOL}.\")\n","inputs/config.json": "{\n  \"frame\": \"local-metre schematic: x = distance inward from the east edge proxy, y = distance south from the north edge proxy\",\n  \"provenance_only\": \"Harbour Road Garden, OpenStreetMap way 27059126 (ID cited for location; no coordinates are loaded)\",\n  \"window_m\": 6.0,\n  \"seat_long_m\": 2.0,\n  \"seat_short_m\": 1.0,\n  \"seat_centre_from_north_m\": 3.0,\n  \"widths_m\": [0.75, 1.0, 1.5],\n  \"base_width_m\": 1.0,\n  \"c1_m\": 0.30,\n  \"c2_m\": 1.50,\n  \"s_pct\": 60.0,\n  \"edge_proxies\": [\"east\", \"north\"],\n  \"sweep_step_m\": 0.05,\n  \"status\": \"Window C, edge proxies, w, c1, c2 and s are researcher assumptions; none is a measured path or a comfort standard\"\n}\n","inputs/variants.csv": "variant,position,orientation,long_axis,centre_from_east_m\nS1,Near,parallel,N-S,2.2\nS2,Near,perpendicular,E-W,2.2\nS3,Centre,parallel,N-S,3.2\nS4,Centre,perpendicular,E-W,3.2\nS5,Far,parallel,N-S,4.3\nS6,Far,perpendicular,E-W,4.3\n","reference/a3_scenarios_reference.csv": "variant,position,orientation,edge_proxy,w_m,corridor_overlap_m2,band_share_pct,seat_area_m2,inside_window,geometry,passes_rule\nS1,Near,parallel,east,0.75,0.0,55.0,2.0,True,clear,False\nS2,Near,perpendicular,east,0.75,0.0,52.5,2.0,True,clear,False\nS3,Centre,parallel,east,0.75,0.0,0.0,2.0,True,clear,False\nS4,Centre,perpendicular,east,0.75,0.0,2.5,2.0,True,clear,False\nS5,Far,parallel,east,0.75,0.0,0.0,2.0,True,clear,False\nS6,Far,perpendicular,east,0.75,0.0,0.0,2.0,True,clear,False\nS1,Near,parallel,east,1.0,0.0,80.0,2.0,True,clear,True\nS2,Near,perpendicular,east,1.0,0.0,60.0,2.0,True,clear,True\nS3,Centre,parallel,east,1.0,0.0,0.0,2.0,True,clear,False\nS4,Centre,perpendicular,east,1.0,0.0,15.0,2.0,True,clear,False\nS5,Far,parallel,east,1.0,0.0,0.0,2.0,True,clear,False\nS6,Far,perpendicular,east,1.0,0.0,0.0,2.0,True,clear,False\nS1,Near,parallel,east,1.5,0.0,90.0,2.0,True,clear,True\nS2,Near,perpendicular,east,1.5,0.3,60.0,2.0,True,conflict,False\nS3,Centre,parallel,east,1.5,0.0,30.0,2.0,True,clear,False\nS4,Centre,perpendicular,east,1.5,0.0,40.0,2.0,True,clear,False\nS5,Far,parallel,east,1.5,0.0,0.0,2.0,True,clear,False\nS6,Far,perpendicular,east,1.5,0.0,0.0,2.0,True,clear,False\nS1,Near,parallel,north,0.75,0.0,12.5,2.0,True,clear,False\nS2,Near,perpendicular,north,0.75,0.0,0.0,2.0,True,clear,False\nS3,Centre,parallel,north,0.75,0.0,12.5,2.0,True,clear,False\nS4,Centre,perpendicular,north,0.75,0.0,0.0,2.0,True,clear,False\nS5,Far,parallel,north,0.75,0.0,12.5,2.0,True,clear,False\nS6,Far,perpendicular,north,0.75,0.0,0.0,2.0,True,clear,False\nS1,Near,parallel,north,1.0,0.0,25.0,2.0,True,clear,False\nS2,Near,perpendicular,north,1.0,0.0,0.0,2.0,True,clear,False\nS3,Centre,parallel,north,1.0,0.0,25.0,2.0,True,clear,False\nS4,Centre,perpendicular,north,1.0,0.0,0.0,2.0,True,clear,False\nS5,Far,parallel,north,1.0,0.0,25.0,2.0,True,clear,False\nS6,Far,perpendicular,north,1.0,0.0,0.0,2.0,True,clear,False\nS1,Near,parallel,north,1.5,0.0,50.0,2.0,True,clear,False\nS2,Near,perpendicular,north,1.5,0.0,50.0,2.0,True,clear,False\nS3,Centre,parallel,north,1.5,0.0,50.0,2.0,True,clear,False\nS4,Centre,perpendicular,north,1.5,0.0,50.0,2.0,True,clear,False\nS5,Far,parallel,north,1.5,0.0,50.0,2.0,True,clear,False\nS6,Far,perpendicular,north,1.5,0.0,50.0,2.0,True,clear,False\n"}
here = Path.cwd()
if (here / 'a3src' / 'a3_workflow.py').exists():
    ROOT = here
    print('Using repository files in', ROOT)
else:
    ROOT = here / 'A3_rerun'
    for rel, text in FILES.items():
        path = ROOT / rel
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(text, encoding='utf8')
    (ROOT / 'a3src' / '__init__.py').write_text('', encoding='utf8')
    print('Wrote embedded files to', ROOT)
print((ROOT / 'inputs' / 'config.json').read_text(encoding='utf8'))

## 3. Run the workflow
36 scenarios, the summaries, the position sweep, 25 internal checks, the GeoPandas cross-check and the figures.

In [ ]:
import subprocess, sys
r = subprocess.run([sys.executable, str(ROOT / 'a3_osm_anchor.py')], cwd=str(ROOT), capture_output=True, text=True)
print(r.stdout)
if r.returncode != 0:
    print(r.stderr[-3000:])
    raise RuntimeError('The workflow reported a failed check or an error; see the output above.')

## 4. Reproduction check
Compares this run with the reference results. Expected: **REPRODUCED**.

In [ ]:
r = subprocess.run([sys.executable, str(ROOT / 'check_reproduction.py')], cwd=str(ROOT), capture_output=True, text=True)
print(r.stdout, r.stderr)
assert r.returncode == 0, 'Results differ from the reference (expected only if you edited the inputs)'

## 5. Results

In [ ]:
import pandas as pd
from IPython.display import display, Image
pd.set_option('display.max_colwidth', 120)
res = pd.read_csv(ROOT / 'outputs' / 'a3_scenarios.csv')
print('Band share (%) by variant, edge proxy and corridor half-width')
display(res.pivot_table(index='variant', columns=['edge_proxy', 'w_m'], values='band_share_pct'))
print('Corridor overlap (m²)')
display(res.pivot_table(index='variant', columns=['edge_proxy', 'w_m'], values='corridor_overlap_m2'))
display(pd.read_csv(ROOT / 'outputs' / 'a3_variant_summary.csv'))

In [ ]:
display(pd.read_csv(ROOT / 'outputs' / 'a3_checks.csv'))

In [ ]:
for name in ['fig_six_placements.png', 'fig_base_comparison_east.png', 'fig_base_comparison_north.png', 'fig_width_sensitivity.png']:
    display(Image(filename=str(ROOT / 'outputs' / name), width=860))

## 6. Position sweep (exploratory)
A parallel seat moved in 0.05 m steps along the east proxy. Rows shown pass all three tested widths.

In [ ]:
sw = pd.read_csv(ROOT / 'outputs' / 'a3_position_sweep.csv')
display(sw[sw.passes_all_widths])

## 7. Change an assumption
Edit `inputs/config.json` or `inputs/variants.csv` under the folder printed in section 2, then run sections 3–6 again. The hand-calculated checks and the reference comparison assume the default inputs, so they will then report differences; that is expected.